# Perplexity from scratch

Computes perplexity three ways on a small synthetic text corpus and compares
the results: non-overlapping windows, sliding windows with stride, and a
closed-form cross-check based on the well-known ``PPL = exp(loss)`` identity.

The notebook uses a tiny character-level language model (trained for a few
steps on the corpus itself so that its perplexity is finite) instead of a
large HuggingFace model. This keeps the whole notebook CPU-safe,
deterministic, and under 60 s, while still exercising the mechanics of
sliding-window PPL that matter for production eval harnesses.

**Learning objectives**
- Derive perplexity from cross-entropy loss.
- Implement sliding-window PPL with overlap (Radford 2019 style).
- See why stride < window reduces PPL vs non-overlapping windows.

**Papers**: Radford et al. 2019 (GPT-2, Appendix C on sliding-window PPL).
**Hardware**: CPU. No model downloads; no external datasets.
**Estimated runtime**: ≈1 min on CPU.


In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "scoring" / "harness.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F

from llm_infra_lab._utils import set_seed
from scoring.harness import Scorer

set_seed(0)
s = Scorer("06_eval_01_perplexity_from_scratch")


In [ ]:
TEXT = (
    "the quick brown fox jumps over the lazy dog. "
    "perplexity is defined as the exponential of the cross-entropy loss. "
    "a language model assigns a probability distribution over the next token. "
    "sliding-window evaluation reduces bias from hard document boundaries. "
    "non-overlapping windows give a pessimistic estimate because early tokens in each window see little context. "
    "tokens sampled with a fair coin have perplexity equal to the alphabet size. "
) * 12

# Char-level vocabulary — deterministic mapping.
CHARS = sorted(set(TEXT))
STOI = {c: i for i, c in enumerate(CHARS)}
ITOS = {i: c for c, i in STOI.items()}
VOCAB = len(CHARS)
print(f"text length={len(TEXT)} vocab={VOCAB}")
IDS = torch.tensor([STOI[c] for c in TEXT], dtype=torch.long)


class CharLM(nn.Module):
    def __init__(self, vocab: int, d_model: int = 64, context: int = 32) -> None:
        super().__init__()
        self.context = context
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(context, d_model)
        self.enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=128,
            batch_first=True, activation="gelu",
        )
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        h = self.emb(x) + self.pos(pos)[None]
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        h = self.enc(h, src_mask=mask, is_causal=True)
        return self.head(self.ln(h))


set_seed(0)
model = CharLM(VOCAB, d_model=64, context=32)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
# A few epochs of training so perplexity is finite and stable.
model.train()
for step in range(60):
    i = step % (len(IDS) - 33)
    batch = IDS[i : i + 33].unsqueeze(0)
    x, y = batch[:, :-1], batch[:, 1:]
    logits = model(x)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), y.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
model.eval()
print(f"final training loss: {loss.item():.3f}")


In [ ]:
@torch.inference_mode()
def perplexity_non_overlapping(ids: torch.Tensor, ctx: int) -> float:
    '''Evaluate ``len(ids)`` tokens in chunks of ``ctx`` with no overlap.'''
    total_nll = 0.0
    total_tok = 0
    for i in range(0, len(ids) - 1, ctx):
        chunk = ids[i : i + ctx + 1]
        if len(chunk) < 2:
            break
        x, y = chunk[:-1].unsqueeze(0), chunk[1:]
        logits = model(x)[0]
        loss = F.cross_entropy(logits, y, reduction="sum")
        total_nll += loss.item()
        total_tok += len(y)
    return math.exp(total_nll / total_tok)


@torch.inference_mode()
def perplexity_sliding(ids: torch.Tensor, ctx: int, stride: int) -> float:
    '''Sliding-window perplexity: overlap contexts, only score the final ``stride`` tokens.'''
    total_nll = 0.0
    total_tok = 0
    prev_end = 0
    for begin in range(0, len(ids), stride):
        end = min(begin + ctx, len(ids))
        if end - begin < 2:
            break
        chunk = ids[begin:end]
        x = chunk[:-1].unsqueeze(0)
        y = chunk[1:]
        logits = model(x)[0]
        # Score only the tokens from prev_end up to end.
        score_from = max(0, prev_end - begin - 1)
        if score_from >= len(y):
            prev_end = end
            continue
        loss = F.cross_entropy(logits[score_from:], y[score_from:], reduction="sum")
        total_nll += loss.item()
        total_tok += len(y) - score_from
        prev_end = end
        if end == len(ids):
            break
    return math.exp(total_nll / total_tok)


@torch.inference_mode()
def perplexity_closed_form(ids: torch.Tensor) -> float:
    '''Evaluate the full sequence as one giant causal pass (reference value).'''
    # Pads the context to the model's training context and scores whatever fits.
    ctx = model.context
    # Truncate to ctx+1 tokens — this sets an upper bound on how much text we can score.
    chunk = ids[: ctx + 1]
    x, y = chunk[:-1].unsqueeze(0), chunk[1:]
    logits = model(x)[0]
    return math.exp(F.cross_entropy(logits, y, reduction="mean").item())


CTX = 32
ppl_nonovl = perplexity_non_overlapping(IDS, ctx=CTX)
ppl_slide  = perplexity_sliding(IDS, ctx=CTX, stride=CTX // 2)
ppl_ref    = perplexity_closed_form(IDS)

print(f"non-overlapping PPL: {ppl_nonovl:.3f}")
print(f"sliding (stride=ctx/2) PPL: {ppl_slide:.3f}")
print(f"closed-form single-pass PPL: {ppl_ref:.3f}")


In [ ]:
# Sliding-window should *not* be higher than non-overlapping: each token is
# scored with strictly more left context.
s.check(
    "sliding_ppl_no_worse_than_non_overlapping",
    lambda: ppl_slide <= ppl_nonovl + 1e-6,
    msg=f"sliding={ppl_slide:.3f} non-overlapping={ppl_nonovl:.3f}",
)
# All PPLs must be finite and bounded above by the vocabulary size (uniform
# random would give exactly VOCAB).
s.check(
    "ppl_bounded_by_vocab",
    lambda: max(ppl_nonovl, ppl_slide, ppl_ref) < VOCAB + 1,
    msg=f"max={max(ppl_nonovl, ppl_slide, ppl_ref):.3f} vocab={VOCAB}",
)
s.check(
    "ppl_positive_and_finite",
    lambda: all(math.isfinite(v) and v > 1.0 for v in [ppl_nonovl, ppl_slide, ppl_ref]),
    msg=f"values=({ppl_nonovl:.3f}, {ppl_slide:.3f}, {ppl_ref:.3f})",
)
# Determinism: running the same evaluation twice should give identical results.
ppl_nonovl2 = perplexity_non_overlapping(IDS, ctx=CTX)
s.assert_close("ppl_deterministic", actual=ppl_nonovl2, expected=ppl_nonovl, rtol=1e-5)
# Relationship between PPL and NLL: ppl = exp(mean_nll) so log-ppl should track
# the model's training loss within the same order of magnitude.
s.check(
    "ppl_reasonable_relative_to_training_loss",
    lambda: math.log(ppl_nonovl) < 2 * loss.item() + 2,
    msg=f"log(ppl_nonovl)={math.log(ppl_nonovl):.3f} train_loss={loss.item():.3f}",
)


In [ ]:
s.summary()
s.save()
